# Writing Exams

- 이메일 답장하기
- 제시문 내용 요약하기
- 자신의 의견쓰기



In [ ]:
import json
from typing import List, Union

from tqdm.notebook import tqdm
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser, CommaSeparatedListOutputParser
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
import pandas as pd



import os


In [2]:
model = ChatOpenAI(model="gpt-5.6-luna")

## 이메일 답장하기

### 가상의 이메일 생성하기

In [3]:
def build_text_sampling_chain(desc):
    prompt_template = PromptTemplate.from_template(template=desc)
    chain = prompt_template | model | StrOutputParser()
    return chain

In [4]:
email_gen_chain = build_text_sampling_chain(desc="영어 Writing 시험에서 이메일에 답장하기에 등장 할 법한 가상의 영어 이메일 하나 만들어줘. 이름 같은 것들도 가상으로 만들어서 채워줘. 영어로")

In [5]:
email_gen_chain.invoke({})

'**Subject: Plans for the School Charity Fair**\n\n**From:** Emily Carter <emily.carter@example.com>  \n**To:** Daniel Kim <daniel.kim@example.com>  \n**Date:** October 14, 2026  \n\nHi Daniel,\n\nHow are you? I’m writing to ask for your help with the school charity fair next month. Our class is planning a food stall to raise money for the local animal shelter.\n\nCould you help us on Saturday, November 21? We need volunteers to prepare food, decorate the stall, and sell snacks during the event. If you are available, please let me know what kind of food you would like to make. Also, do you have any ideas for making our stall more attractive?\n\nThe fair will begin at 10 a.m. and finish at 3 p.m. I hope you can join us!\n\nBest wishes,  \n**Emily**'

### 답장 평가하기

In [6]:
def build_eval_chain(instruction, reason_desc, score_desc):
    class Evaluation(BaseModel):
        reason: str = Field(description=reason_desc)
        score: int = Field(description=score_desc)
    
    parser = JsonOutputParser(pydantic_object=Evaluation)
    format_instructions = parser.get_format_instructions()
    
    human_prompt_template = HumanMessagePromptTemplate.from_template(
                                "# Instruction: {instruction}\n"
                                "# Context: {context}\n"
                                "# User: {input}\n"
                                "{format_instructions}",
                                partial_variables={"instruction": instruction,
                                                   "format_instructions": format_instructions})
    
    prompt = ChatPromptTemplate.from_messages(
        [
            human_prompt_template,
        ])
    eval_chain = prompt | model | parser
    return eval_chain

In [7]:
email_eval_chain = build_eval_chain(instruction="User의 응답이 Context의 이메일에 대한 적절한 응답인지 추론하고 평가하라",
                                    reason_desc="User의 응답이 Context의 이메일에 대한 적절한 응답인지에 대한 추론",
                                    score_desc="User의 응답이 Context의 이메일에 대한 적절한 응답인지에 대한 점수, 0~10점")

In [8]:
email = email_gen_chain.invoke({})

In [9]:
email

'**From:** Emily Carter <emily.carter@example.com>  \n**To:** Daniel Kim <daniel.kim@example.com>  \n**Subject:** Plans for Our School Cultural Festival  \n\nHi Daniel,\n\nHow are you? I’m writing because our school is holding a cultural festival next month, and I’m helping to organize the international food booth. Since you visited several countries last year, I thought you might have some good ideas.\n\nWhat kind of food do you think we should prepare? We also need to decide how to decorate the booth and what activities to offer visitors. Could you help us on Saturday, October 12? The festival will begin at 10 a.m., but we need to arrive earlier to get everything ready.\n\nPlease let me know if you can come and what you think we should do. I’m looking forward to hearing from you!\n\nBest wishes,  \nEmily'

In [10]:
user_answer = "I would be delighted to attend the annual Book Club meeting on Saturday, June 15th, and look forward to seeing everyone."

In [11]:
eval_result = email_eval_chain.invoke({"context": email, "input": user_answer})

In [12]:
eval_result

{'reason': '사용자의 응답은 학교 문화 축제의 국제 음식 부스 준비에 관한 이메일과 관련이 없습니다. 요청된 10월 12일 참석 여부, 음식 아이디어, 부스 장식 및 활동에 대한 답변 대신 6월 15일 연례 독서 모임 참석 의사만 밝히고 있습니다.',
 'score': 0}

## 제시문 내용 요약하기

### 무작위 글 생성

In [13]:
text_gen_chain = build_text_sampling_chain(desc="영어 Writing 시험에서 단락 요약하기에 등장 할 법한 가상의 영어 단락 하나 만들어줘. 이름 같은 것들도 가상으로 만들어서 채워줘. 영어로")

In [14]:
text = text_gen_chain.invoke({})
text

'In the fictional town of Bellmare, a high school student named Liora Venn started a community garden on an unused piece of land behind the local library. At first, many residents doubted that the project would succeed because the soil was dry and the neighborhood had little experience with gardening. However, Liora organized weekend workshops, invited an environmental scientist named Dr. Orin Vale to teach residents about sustainable farming, and persuaded local stores to donate seeds and tools. Within a year, the garden was producing vegetables for nearby families and a small food bank. More importantly, it brought people of different ages together and encouraged them to take greater responsibility for their community. The success of Bellmare’s garden showed that even a small project can create meaningful social change when people cooperate and remain committed to a common goal.'

### 요약 평가하기

In [15]:
summarization_eval_chain = build_eval_chain(instruction="User의 응답이 Context에 대한 적절한 요약인지 추론하고 평가하라",
                                            reason_desc="User의 응답이 Context에 대한 적절한 요약인지에 대한 추론",
                                            score_desc="User의 응답이 Context에 대한 적절한 요약인지에 대한 점수, 0~10점")

In [16]:
user_answer = "Emily Johnson spent a memorable week visiting her cousin Michael in Seattle, where he introduced her to local spots like Pike Place Market and nearby mountains, resulting in new experiences and happy memories."

In [17]:
summarization_eval_chain.invoke({"context": text,
                                 "input": user_answer})

{'reason': '사용자의 응답은 벨마레의 공동체 정원 조성, 리오라 벤의 노력, 주민 협력과 사회적 변화에 대한 Context의 내용을 전혀 반영하지 않고, 에밀리 존슨의 시애틀 여행이라는 unrelated한 내용을 제시하고 있습니다.',
 'score': 0}

## 자신의 의견쓰기

### 무작위 이슈 생성


In [18]:
issue_gen_chain = build_text_sampling_chain(desc="영어 Writing 시험에서 자신의 의견쓰기에 등장 할 법한 무작위 이슈 영어 단락 하나 만들어줘. 이름 같은 것들도 가상으로 만들어서 채워줘. 영어로")

In [19]:
issue = issue_gen_chain.invoke({})
issue

'In my opinion, high schools should allow students to use their phones during lunch breaks. Some people, such as the fictional principal Mr. Daniel Brooks, worry that phones will make students less social. However, I believe that limited phone use can be beneficial. Students can contact their parents, check important information, or relax by listening to music after a stressful morning. In addition, teaching students when and how to use phones responsibly is more realistic than banning them completely. For example, my imaginary classmate, Emily Carter, uses her phone during lunch to organize her homework and send messages to her younger brother. Of course, schools should prohibit phones during lessons and examinations. With clear rules, students can enjoy the advantages of technology without allowing it to interfere with their education.'

### 의견 평가하기

In [20]:
opinion_eval_chain = build_eval_chain(instruction="User의 응답이 Context에 대한 적절한 의견 주장인지 추론하고 평가하라",
                                      reason_desc="User의 응답이 Context에 대한 적절한 의견 주장인지 대한 추론",
                                      score_desc="User의 응답이 Context에 대한 적절한 의견 주장인지에 대한 점수, 0~10점")

In [21]:
user_answer = "While mandatory school uniforms may mitigate issues of peer pressure and cost, they fundamentally limit students' important avenues for self-expression and the development of personal identity."

In [22]:
opinion_eval_chain.invoke({"context": issue,
                           "input": user_answer})

{'reason': '사용자의 주장은 의무 교복이 학생의 자기표현과 정체성 형성에 미치는 영향에 관한 것으로, 점심시간 휴대전화 사용 허용 여부를 다루는 Context의 핵심 주장과 직접적인 관련이 없습니다.',
 'score': 0}